# HumanForYou — Preprocessing & Feature Engineering

**Objectif :** Transformer les données brutes en un jeu de données propre et prêt pour la modélisation.

Les étapes :
1. Fusion des 4 fichiers sources
2. Suppression des colonnes inutiles
3. Extraction de features depuis les données de badgeage
4. Imputation des valeurs manquantes (médiane)
5. Encodage des variables catégorielles
6. Normalisation des variables numériques
7. Sauvegarde du jeu de données final

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RAW  = '../data/raw/'
PROC = '../data/processed/'

---
## 1. Chargement des données

In [2]:
general  = pd.read_csv(RAW + 'general_data.csv')
survey   = pd.read_csv(RAW + 'employee_survey_data.csv')
in_time  = pd.read_csv(RAW + 'in_time.csv',  index_col=0)
out_time = pd.read_csv(RAW + 'out_time.csv', index_col=0)

print(f"general_data : {general.shape}")
print(f"survey_data  : {survey.shape}")
print(f"in_time      : {in_time.shape}")
print(f"out_time     : {out_time.shape}")

general_data : (4410, 24)
survey_data  : (4410, 4)
in_time      : (4410, 261)
out_time     : (4410, 261)


---
## 2. Suppression des colonnes inutiles

Identifiées lors de l'EDA : ces 3 colonnes ont la même valeur pour tous les employés → elles n'apportent **aucune information** au modèle.

In [3]:
cols_a_supprimer = ['EmployeeCount', 'Over18', 'StandardHours']

print("Valeurs uniques avant suppression :")
for col in cols_a_supprimer:
    print(f"  {col} → {general[col].unique()}")

general = general.drop(columns=cols_a_supprimer)
print(f"\nDimensions après suppression : {general.shape}")

Valeurs uniques avant suppression :
  EmployeeCount → [1]
  Over18 → ['Y']
  StandardHours → [8]

Dimensions après suppression : (4410, 21)


---
## 3. Feature engineering — données de badgeage

Les fichiers `in_time` et `out_time` contiennent les horaires d'arrivée et de départ pour chaque jour de 2015 (261 colonnes-dates).  
On ne peut pas donner 261 colonnes brutes à un modèle ML — on va les **résumer en quelques indicateurs parlants**.

### Pourquoi ces indicateurs ?
- **Heures travaillées en moyenne** → un employé épuisé ou sous-chargé risque plus de partir
- **Jours d'absence** → un employé absent souvent est peut-être désengagé
- **Jours de dépassement (> 8h)** → surcharge chronique = risque de burn-out
- **Variabilité des horaires** → instabilité du rythme de travail
- **Heure d'arrivée moyenne** → indicateur indirect d'engagement

In [4]:
# Convertir les colonnes en datetime
in_dt  = in_time.apply(pd.to_datetime, errors='coerce')
out_dt = out_time.apply(pd.to_datetime, errors='coerce')

# Calculer la durée travaillée chaque jour (en heures)
work_hours = (out_dt - in_dt).apply(lambda col: col.dt.total_seconds() / 3600)

# Construire le tableau de features
badge = pd.DataFrame()
badge['EmployeeID']         = general['EmployeeID'].values
badge['avg_hours_per_day']  = work_hours.mean(axis=1).values
badge['std_hours_per_day']  = work_hours.std(axis=1).values       # variabilité
badge['days_absent']        = work_hours.isna().sum(axis=1).values # jours sans badge
badge['days_over_8h']       = (work_hours > 8).sum(axis=1).values  # jours > 8h
badge['avg_arrival_hour']   = in_dt.apply(
    lambda col: col.dt.hour + col.dt.minute / 60
).mean(axis=1).values

print(badge.head())
print()
print(badge.describe().round(2))

   EmployeeID  avg_hours_per_day  std_hours_per_day  days_absent  \
0           1           7.373651           0.283224           29   
1           2           7.718969           0.313351           25   
2           3           7.013240           0.311551           19   
3           4           7.193678           0.284133           26   
4           5           8.006175           0.300656           16   

   days_over_8h  avg_arrival_hour  
0             0          9.993032  
1            42          9.980720  
2             0         10.016598  
3             0          9.973830  
4           115          9.990068  

       EmployeeID  avg_hours_per_day  std_hours_per_day  days_absent  \
count     4410.00            4410.00            4410.00      4410.00   
mean      2205.50               7.70               0.30        24.73   
std       1273.20               1.34               0.01         5.50   
min          1.00               5.95               0.25        13.00   
25%       1103

---
## 4. Fusion des 4 sources de données

On joint tout sur la clé commune `EmployeeID`.

In [5]:
df = general.merge(survey, on='EmployeeID', how='left')
df = df.merge(badge,  on='EmployeeID', how='left')

print(f"Dimensions après fusion : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

Dimensions après fusion : (4410, 29)
Colonnes : ['Age', 'Attrition', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeID', 'Gender', 'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'avg_hours_per_day', 'std_hours_per_day', 'days_absent', 'days_over_8h', 'avg_arrival_hour']


---
## 5. Imputation des valeurs manquantes

**Stratégie : imputation par la médiane** pour toutes les colonnes numériques.

La médiane est préférée à la moyenne car elle est **robuste aux valeurs extrêmes** (un salaire très élevé ne va pas fausser le remplacement).

> C'est la même stratégie que dans votre Workshop Boucle 1 sur le dataset immobilier californien.

In [6]:
# Valeurs manquantes avant imputation
na_avant = df.isna().sum()
na_avant = na_avant[na_avant > 0]
print("Valeurs manquantes AVANT imputation :")
print(na_avant)

# Imputation par la médiane sur toutes les colonnes numériques
num_cols = df.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    if df[col].isna().sum() > 0:
        mediane = df[col].median()
        df[col] = df[col].fillna(mediane)
        print(f"  → {col} : NA remplacés par la médiane ({mediane:.2f})")

print()
print(f"Valeurs manquantes APRÈS imputation : {df.isna().sum().sum()}")

Valeurs manquantes AVANT imputation :
NumCompaniesWorked         19
TotalWorkingYears           9
EnvironmentSatisfaction    25
JobSatisfaction            20
WorkLifeBalance            38
dtype: int64
  → NumCompaniesWorked : NA remplacés par la médiane (2.00)
  → TotalWorkingYears : NA remplacés par la médiane (10.00)
  → EnvironmentSatisfaction : NA remplacés par la médiane (3.00)
  → JobSatisfaction : NA remplacés par la médiane (3.00)
  → WorkLifeBalance : NA remplacés par la médiane (3.00)

Valeurs manquantes APRÈS imputation : 0


---
## 6. Encodage des variables catégorielles

Les algorithmes ML ne comprennent pas les mots (`'Male'`, `'Female'`…), ils travaillent uniquement avec des **chiffres**.

On distingue deux cas :
- **Variables binaires** (2 valeurs) → `LabelEncoder` : on remplace simplement par 0/1
- **Variables à plusieurs catégories** → `pd.get_dummies` (One-Hot Encoding) : on crée une colonne 0/1 par catégorie

### Pourquoi One-Hot pour les variables multi-catégories ?
Si on encodait `JobRole` avec des chiffres (0, 1, 2…), le modèle pourrait penser que le rôle n°3 est "plus grand" que le rôle n°1, ce qui n'a aucun sens. One-Hot évite ce problème.

In [7]:
# --- Variable cible : Attrition ---
# On encode Yes=1, No=0
df['Attrition'] = (df['Attrition'] == 'Yes').astype(int)
print("Attrition encodée :")
print(df['Attrition'].value_counts())

# --- Variables binaires (LabelEncoder) ---
binary_cols = ['Gender']  # Male/Female
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
    print(f"\n{col} encodé : {dict(zip(le.classes_, le.transform(le.classes_)))}")

# --- Variables multi-catégories (One-Hot Encoding) ---
ohe_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print(f"\nDimensions après encodage : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

Attrition encodée :
Attrition
0    3699
1     711
Name: count, dtype: int64

Gender encodé : {'Female': 0, 'Male': 1}

Dimensions après encodage : (4410, 43)
Colonnes : ['Age', 'Attrition', 'DistanceFromHome', 'Education', 'EmployeeID', 'Gender', 'JobLevel', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'avg_hours_per_day', 'std_hours_per_day', 'days_absent', 'days_over_8h', 'avg_arrival_hour', 'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely', 'Department_Research & Development', 'Department_Sales', 'EducationField_Life Sciences', 'EducationField_Marketing', 'EducationField_Medical', 'EducationField_Other', 'EducationField_Technical Degree', 'JobRole_Human Resources', 'JobRole_Laboratory Technician', 'JobRole_Manager', 'JobRole_Manufacturing Director', 'JobR

---
## 7. Normalisation des variables numériques

Certains algorithmes (régression logistique, SVM, perceptron) sont **sensibles aux échelles** : si une variable va de 0 à 100 000 (salaire) et une autre de 1 à 5 (niveau d'éducation), le modèle va surpondérer le salaire.

**StandardScaler** transforme chaque variable pour qu'elle ait :
- une **moyenne de 0**
- un **écart-type de 1**

C'est la **standardisation** vue dans votre cours Boucle 1.

> Note : on ne normalise **pas** la variable cible `Attrition` ni les colonnes One-Hot (déjà en 0/1).

In [8]:
# Colonnes à ne PAS normaliser
cols_exclure = ['EmployeeID', 'Attrition']
# On exclut aussi les colonnes One-Hot (booléens 0/1)
ohe_generated = [c for c in df.columns if any(c.startswith(base + '_') for base in
                 ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus'])]
cols_exclure += ohe_generated

# Colonnes numériques à normaliser
cols_a_normaliser = [c for c in df.select_dtypes(include='number').columns
                     if c not in cols_exclure]

print("Colonnes normalisées :", cols_a_normaliser)

scaler = StandardScaler()
df[cols_a_normaliser] = scaler.fit_transform(df[cols_a_normaliser])

print("\nAperçu après normalisation :")
df[cols_a_normaliser[:5]].describe().round(2)

Colonnes normalisées : ['Age', 'DistanceFromHome', 'Education', 'Gender', 'JobLevel', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance', 'avg_hours_per_day', 'std_hours_per_day', 'days_absent', 'days_over_8h', 'avg_arrival_hour']

Aperçu après normalisation :


,Age,DistanceFromHome,Education,Gender,JobLevel
count,4410.00,4410.00,4410.00,4410.00,4410.00
mean,-0.00,0.00,0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00
min,-2.07,-1.01,-1.87,-1.22,-0.96
25%,-0.76,-0.89,-0.89,-1.22,-0.96
50%,-0.10,-0.27,0.09,0.82,-0.06
75%,0.67,0.59,1.06,0.82,0.85
max,2.53,2.44,2.04,0.82,2.65


---
## 8. Vérification finale et sauvegarde

In [9]:
# Vérification : aucune valeur manquante, tous des nombres
print("=== Vérification finale ===")
print(f"Shape             : {df.shape}")
print(f"Valeurs manquantes: {df.isna().sum().sum()}")
print(f"Types uniques     : {df.dtypes.unique()}")
print(f"Distribution cible:")
print(df['Attrition'].value_counts())

df.head(3)

=== Vérification finale ===
Shape             : (4410, 43)
Valeurs manquantes: 0
Types uniques     : [dtype('float64') dtype('int32') dtype('int64') dtype('bool')]
Distribution cible:
Attrition
0    3699
1     711
Name: count, dtype: int64


,Age,Attrition,DistanceFromHome,Education,EmployeeID,Gender,JobLevel,MonthlyIncome,NumCompaniesWorked,PercentSalaryHike,...,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
0,1.541369,0,-0.393938,-0.891688,1,-1.224745,-0.961486,1.405136,-0.678464,-1.150554,...,False,False,False,False,False,False,False,False,True,False
1,-0.648668,1,0.099639,-1.868426,2,-1.224745,-0.961486,-0.491661,-1.079486,2.129306,...,False,False,False,False,False,True,False,False,False,True
2,-0.539166,0,0.963398,1.061787,3,0.816497,1.749610,2.725053,-0.678464,-0.057267,...,False,False,False,False,False,False,True,False,True,False


In [10]:
# Sauvegarde
df.to_csv(PROC + 'dataset_final.csv', index=False)
print(f"Fichier sauvegardé : {PROC}dataset_final.csv")
print(f"Shape : {df.shape}")

Fichier sauvegardé : ../data/processed/dataset_final.csv
Shape : (4410, 43)


---
## 9. Résumé des transformations appliquées

| Étape | Action | Pourquoi |
|-------|--------|----------|
| Colonnes inutiles | Suppression de `EmployeeCount`, `Over18`, `StandardHours` | Valeur identique pour tous → aucune info |
| Badgeage | Extraction de 5 features synthétiques | Résumer 261 colonnes en indicateurs utiles |
| Fusion | Merge des 4 fichiers sur `EmployeeID` | Avoir toutes les infos sur un seul tableau |
| Valeurs manquantes | Imputation par la médiane | Robuste aux valeurs extrêmes |
| Encodage binaire | `LabelEncoder` pour `Gender` | Convertir texte → 0/1 |
| Encodage multi | `get_dummies` pour 5 colonnes | Éviter les faux ordres entre catégories |
| Normalisation | `StandardScaler` sur les numériques | Éviter que les grandes échelles dominent |

**Prochaine étape : notebook 03 — modélisation (régression logistique, arbre de décision, Random Forest)**